# 语义分割


## 简介
语义分割（Semantic segmentation），将图片中每个像素分类到对应类别

![segmentation](./images/segmentation.svg)

## 分类
- 语义分割:
  - 将图片中每个像素分类到对应类别
  - 不区分同一类别的不同实例
- 实例分割:
  - 将图片中每个像素分类到对应实例
  - 区分同一类别的不同实例
- 图像分割:
  - 将图片分成不同的区域
  - 不指明类别和实例，只表示相关性


## 实现


In [1]:
%matplotlib inline
import os
import torch
import torchvision
import util
util.data.DATA_HUB['voc2012'] = (util.data.DATA_URL + 'VOCtrainval_11-May-2012.tar',
                           '4e443f8a2eca6b1dac8a6c57641b67dd40621a49')

voc_dir = util.data.download_extract('voc2012', 'VOCdevkit/VOC2012')

/root/autodl-tmp/envs/daily_learning/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


正在从https://d2l-data.s3-accelerate.amazonaws.com/VOCtrainval_11-May-2012.tar下载./data/VOCtrainval_11-May-2012.tar...


KeyboardInterrupt: 

读入内存


In [ ]:
def read_voc_images(voc_dir, is_train=True):
    """读取VOC数据集中的图像和标签"""
    txt_fname = os.path.join(voc_dir, 'ImageSets', 'Segmentation',
                             'train.txt' if is_train else 'val.txt')
    mode = torchvision.io.image.ImageReadMode.RGB
    with open(txt_fname, 'r') as f:
        images = f.read().split()
    features, labels = [], []
    for i, fname in enumerate(images):
        features.append(torchvision.io.read_image(os.path.join(
            voc_dir, 'JPEGImages', f'{fname}.jpg')))
        labels.append(torchvision.io.read_image(os.path.join( # label 也是图片，每个像素的值是类别索引
            voc_dir, 'SegmentationClass' ,f'{fname}.png'), mode))
    return features, labels

train_features, train_labels = read_voc_images(voc_dir, True)

检查图片格式


In [ ]:
n = 5
imgs = train_features[:n] + train_labels[:n]
imgs = [img.permute(1,2,0) for img in imgs]
util.show_images(imgs, 2, n)

标号映射


In [ ]:
VOC_COLORMAP = [[0, 0, 0], [128, 0, 0], [0, 128, 0], [128, 128, 0],
                [0, 0, 128], [128, 0, 128], [0, 128, 128], [128, 128, 128],
                [64, 0, 0], [192, 0, 0], [64, 128, 0], [192, 128, 0],
                [64, 0, 128], [192, 0, 128], [64, 128, 128], [192, 128, 128],
                [0, 64, 0], [128, 64, 0], [0, 192, 0], [128, 192, 0],
                [0, 64, 128]]

VOC_CLASSES = ['background', 'aeroplane', 'bicycle', 'bird', 'boat',
               'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
               'diningtable', 'dog', 'horse', 'motorbike', 'person',
               'potted plant', 'sheep', 'sofa', 'train', 'tv/monitor']

In [ ]:
def voc_colormap2label():
    """建立从RGB到类别索引的映射"""
    def hashing(colormap):
        return (colormap[0] * 256 + colormap[1]) * 256 + colormap[2]
    colormap2label = torch.zeros(256 ** 3, dtype=torch.uint8) # 存储每个颜色对应的类别索引
    for i, colormap in enumerate(VOC_COLORMAP):
        colormap2label[hashing(colormap)] = i # 将每个颜色映射到对应的类别索引
    return colormap2label

def voc_label_indices(colormap, colormap2label):
    """将彩色标签映射到类别索引"""
    def hashing(colormap):
        return (colormap[:, :, 0] * 256 + colormap[:, :, 1]) * 256 + colormap[:, :, 2]
    colormap = colormap.permute(1, 2, 0).numpy().astype('int32') # 将通道维度移到最后
    idx = hashing(colormap) # 将RGB映射到整数
    return colormap2label[idx] # 返回类别索引

例如


In [ ]:
y = voc_label_indices(train_labels[0], voc_colormap2label())
y[105:115, 130:140], VOC_CLASSES[1] # 显示某个区域类别索引(1)和类别名称(aeroplane)

随机剪裁的图像增广，还需要同样剪裁标签


In [ ]:
def voc_rand_crop(feature, label, height, width):
    """随机裁剪图像和标签"""
    # get_params 返回裁剪窗口的左上角坐标和高宽
    rect = torchvision.transforms.RandomCrop.get_params(feature, (height, width))
    # 通用的参数裁剪图像和标签
    feature = torchvision.transforms.functional.crop(feature, *rect)
    label = torchvision.transforms.functional.crop(label, *rect)
    return feature, label

In [ ]:
imgs = []
for _ in range(n):
    imgs += voc_rand_crop(train_features[0], train_labels[0], 200, 300)
imgs = [img.permute(1,2,0) for img in imgs]
util.show_images(imgs[::2]+imgs[1::2], 2, n) # 显示裁剪后的图像和标签

自定义数据集类


In [ ]:
class VOCSegDataset(torch.utils.data.Dataset):
    """VOC语义分割数据集"""
    def __init__(self, is_train: bool, crop_size: tuple[int, int], voc_dir: str):
        self.transform = torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                          std=[0.229, 0.224, 0.225]) # ImageNet 标准
        self.crop_size = crop_size
        features, labels = read_voc_images(voc_dir, is_train)
        self.features = [self.normalize_image(feat) for feat in self.filter(features)]
        self.labels = self.filter(labels)
        self.colormap2label = voc_colormap2label()
        print(f'read {len(self.features)} examples')

    def normalize_image(self, img):
        """标准化图像"""
        return self.transform(img.float())
    
    def filter(self, imgs):
        """过滤掉小于裁剪尺寸的图像"""
        return [img for img in imgs if (img.shape[1] >= self.crop_size[0] and 
                                        img.shape[2] >= self.crop_size[1])]
    
    def __getitem__(self, idx):
        """返回裁剪后的图像和标签"""
        feature, label = voc_rand_crop(self.features[idx], self.labels[idx],
                                       *self.crop_size)
        return feature.float(), voc_label_indices(label, self.colormap2label)

    def __len__(self):
        """返回数据集大小"""
        return len(self.features)

读取数据集 


In [ ]:
crop_size = (320, 480)
voc_train = VOCSegDataset(is_train=True, crop_size=crop_size, voc_dir=voc_dir)
voc_test = VOCSegDataset(is_train=False, crop_size=crop_size, voc_dir=voc_dir)

In [ ]:
train_iter = torch.utils.data.DataLoader(voc_train, batch_size=64, shuffle=True, drop_last=True, num_workers=0)
for X, Y in train_iter:
    print(X.shape)
    print(Y.shape)
    break

整合所有组件


In [ ]:
def load_data_voc(batch_size, crop_size):
    """加载VOC数据集"""
    voc_dir = util.data.download_extract('voc2012', 'VOCdevkit/VOC2012')
    train_iter = torch.utils.data.DataLoader(
        VOCSegDataset(True, crop_size, voc_dir),
        batch_size=batch_size, 
        shuffle=True, 
        drop_last=True, 
        num_workers=util.data.get_dataloader_workers())
    test_iter = torch.utils.data.DataLoader(
        VOCSegDataset(False, crop_size, voc_dir), 
        batch_size=batch_size, 
        drop_last=True, 
        num_workers=0)
    return train_iter, test_iter